## Part 3: Intent-Based Multi-Agent Orchestration

This is where single-turn RAG becomes conversational AI. We walk through a fully built orchestration system that classifies user intent, routes to specialized subagents, fills required slots through natural conversation, and handles intent switches mid-conversation with clean state resets.

All of the agent code lives in the `agents/` directory in this repo. In this notebook, we import it, inspect how each piece works, and run live demos.

**Key concepts**: intent classification, slot filling, subagent design, conversational state, intent handoff, session scope vs intent scope.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL", "postgresql://root@localhost:26257/workshop?sslmode=disable")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("No OpenAI API key found. Update your .env file.")
else:
    print(f"Database URL: {DATABASE_URL}")
    print(f"OpenAI API Key: {OPENAI_API_KEY[:10]}...")

In [ ]:
import psycopg2

conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = True

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM products")
    count = cur.fetchone()[0]
print(f"Connected. Table 'products' has {count} rows")

## Intent Classification

The first step in any multi-agent system is understanding what the user wants. Our intent classifier takes a user utterance and returns a structured result: the **intent label**, a **confidence score**, and the **reasoning** behind the classification.

This uses Pydantic AI with a structured output type — the LLM is forced to return a valid `IntentResult` object every time.

Supported intents:
- **billing** — pricing, budget, value questions
- **product** — product discovery, comparisons, browsing
- **support** — product details, specifications, sizing help

In [ ]:
from agents.intent_classifier import classify_intent, SUPPORTED_INTENTS

print(f"Supported intents: {SUPPORTED_INTENTS}")

In [ ]:
# Clear product intent
result = await classify_intent("What kind of jackets do you have for men?")
print(f"Intent: {result.intent}")
print(f"Confidence: {result.confidence}")
print(f"Reasoning: {result.reasoning}")

In [ ]:
# Clear support intent
result = await classify_intent("Can you help me find something that fits well for a formal event?")
print(f"Intent: {result.intent}")
print(f"Confidence: {result.confidence}")
print(f"Reasoning: {result.reasoning}")

In [ ]:
# Ambiguous query — notice the lower confidence
result = await classify_intent("Tell me about your stuff")
print(f"Intent: {result.intent}")
print(f"Confidence: {result.confidence}")
print(f"Reasoning: {result.reasoning}")

## Slot Filling

Once we know the intent, we need to collect the required information. Each intent has its own **slot definitions** — required and optional fields that the system needs before it can execute a query.

The slot filler extracts values from the user's message and generates natural clarifying questions for anything that's missing. Critically, it **never hallucinate slot values** — if the user didn't say it, the slot stays empty.

In [ ]:
from agents.slot_filler import extract_slots, SLOT_DEFINITIONS

# Let's look at what each intent needs
for intent, slots in SLOT_DEFINITIONS.items():
    print(f"\n{intent.upper()}:")
    print(f"  Required: {slots['required']}")
    print(f"  Optional: {slots['optional']}")

In [ ]:
# User provides enough info in one message
result = await extract_slots(
    utterance="Show me black dresses in the Ladieswear department",
    intent="product"
)

print(f"Extracted slots: {result.extracted_slots}")
print(f"Missing required: {result.missing_required}")
print(f"Clarifying question: {result.clarifying_question}")

In [ ]:
# User is vague — slot filler asks for what it needs
result = await extract_slots(
    utterance="I'm looking for something nice to wear",
    intent="product"
)

print(f"Extracted slots: {result.extracted_slots}")
print(f"Missing required: {result.missing_required}")
print(f"Clarifying question: {result.clarifying_question}")

In [ ]:
# Multi-turn: user answers the clarifying question
result = await extract_slots(
    utterance="A dress, maybe something formal",
    intent="product",
    existing_slots={"feature_keyword": "nice to wear"}  # carried from previous turn
)

print(f"Extracted slots: {result.extracted_slots}")
print(f"Missing required: {result.missing_required}")
print(f"Clarifying question: {result.clarifying_question}")

## Intent-Specific Subagents

Each intent gets its own subagent with specialized instructions and knowledge. This is more maintainable and accurate than a single monolithic agent trying to handle everything.

Our three subagents:
- **Billing agent** — Focuses on pricing, budget, and value
- **Product agent** — Focuses on product discovery and comparisons
- **Support agent** — Focuses on product details, specifications, and fit

Each subagent takes the filled slots, builds SQL filters from them, retrieves relevant products, and generates a focused answer.

In [ ]:
from agents.subagents.billing_agent import handle_billing_query
from agents.subagents.product_agent import handle_product_query
from agents.subagents.support_agent import handle_support_query

# Test the product subagent directly
response = await handle_product_query(
    conn=conn,
    slots={"contract_type": "Dress", "feature_keyword": "formal elegant"}
)
print(response)

In [ ]:
# Test the support subagent directly
response = await handle_support_query(
    conn=conn,
    slots={"issue_keyword": "warm winter jacket waterproof"}
)
print(response)

## The Orchestrator

The orchestrator ties everything together. On each user turn it:

1. **Classifies intent** — or continues the current one (prevents over-eager reclassification)
2. **Extracts slots** — merges new values into the session state
3. **Checks completeness** — are all required slots filled?
4. **Routes or asks** — if complete, routes to the subagent; if not, asks a clarifying question

It also handles **intent switches**: if the user changes topics mid-conversation, the orchestrator does a clean state reset — current intent, confidence, and filled slots all get wiped. This prevents **intent bleed**.

In [ ]:
from agents.orchestrator import SessionState, process_turn

## Demo 1: Multi-Turn Product Search

Watch the system classify intent, fill slots across multiple turns, and route to the product subagent.

In [ ]:
session = SessionState()

# Turn 1: User starts with a vague product question
response = await process_turn(session, "I'm looking for something to wear", conn)
print(f"\nAssistant: {response}")

In [ ]:
# Turn 2: User provides the missing slot
response = await process_turn(session, "A dress, something for a party", conn)
print(f"\nAssistant: {response}")

In [ ]:
# Inspect the session state
print(f"Current intent: {session.current_intent}")
print(f"Confidence: {session.intent_confidence}")
print(f"Filled slots: {session.filled_slots}")
print(f"Turn count: {session.turn_count}")
print(f"Conversation history: {len(session.conversation_history)} messages")

## Demo 2: Intent Switch Mid-Conversation

This is the critical test. The user starts browsing products, then switches to asking for detailed help. Watch the clean state reset.

In [ ]:
session2 = SessionState()

# Turn 1: Product browsing with all slots
response = await process_turn(
    session2, 
    "Show me men's dark jackets", 
    conn
)
print(f"\nAssistant: {response}")
print(f"\n--- State: intent={session2.current_intent}, slots={session2.filled_slots}")

In [ ]:
# Turn 2: Switch to support — clean state reset
response = await process_turn(
    session2, 
    "Actually, can you help me find something waterproof for hiking?", 
    conn
)
print(f"\nAssistant: {response}")
print(f"\n--- State: intent={session2.current_intent}, slots={session2.filled_slots}")

## Demo 3: Low-Confidence Handling

What happens when the system can't confidently classify the intent? It asks for clarification rather than guessing.

In [ ]:
session3 = SessionState()

# Vague query
response = await process_turn(session3, "I need help", conn)
print(f"\nAssistant: {response}")
print(f"\n--- State: intent={session3.current_intent}, slots={session3.filled_slots}")

In [ ]:
# User clarifies
response = await process_turn(
    session3, 
    "I want to find a warm winter coat in red or burgundy", 
    conn
)
print(f"\nAssistant: {response}")
print(f"\n--- State: intent={session3.current_intent}, slots={session3.filled_slots}")

## Key Takeaways

**What we've seen:**
1. **Intent classification** — Structured output with confidence scoring catches ambiguity early
2. **Slot filling** — Multi-turn extraction without hallucinating values the user never provided
3. **Subagent routing** — Each intent gets specialized handling with its own instructions and filter logic
4. **State management** — Clean intent handoff prevents intent bleed across topic switches
5. **Over-eager reclassification prevention** — Follow-up messages stay in the current intent unless there's a strong signal to switch

**Why this matters:**
- Monolithic agents break down when user needs shift mid-conversation
- Intent-based design is fundamentally different from rank-based routing
- Clean state management prevents intent bleed and slot hallucination

**Architecture recap:**
```
User Utterance → Intent Classifier → Orchestrator → Slot Filler → Subagent → CockroachDB → Response
```

Next up: how to measure whether your system is actually working.

In [ ]:
conn.close()